# Chapter 54: Multimodal Embeddings & Retrieval
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Objective:** Mempelajari topologi ruang metrik bersama (*joint metric space*), membuktikan kesetaraan analitis jarak Euclidean kuadrat dengan kesamaan kosinus pada unit hypersphere, mengimplementasikan algoritma pencarian tetangga terdekat lintas-modal (T2I & I2T), serta mengevaluasi metrik standar industri Recall@K dan Mean Reciprocal Rank (MRR).

### 1. Landasan Geometri Ruang Metrik & Metrik Evaluasi
Kesetaraan Metrik Jarak pada Vektor Ternormalisasi $\ell_2$:
$$\|\hat{\mathbf{u}} - \hat{\mathbf{v}}\|_2^2 = 2(1 - \cos\theta)$$

Metrik Evaluasi Sistem Retrieval:
$$\text{Recall}@K = \frac{1}{N_q} \sum_{i=1}^{N_q} \mathbf{1}(\text{rank}_i \le K), \quad \text{MRR} = \frac{1}{N_q} \sum_{i=1}^{N_q} \frac{1}{\text{rank}_i}$$

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Beroperasi dalam Mode Analisis Algoritmik NumPy murni.")

In [ ]:
# 3. Pembuktian Kesetaraan Geometri & Validasi Evaluasi Seksi 3
def l2_norm(v, eps=1e-12):
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.maximum(n, eps)

# Vektor Uji Sesuai Seksi 3 Part A
u = np.array([1.0, 0.0])
v = np.array([0.6, 0.8])

cos_theta = float(np.dot(u, v))
d_e_sq = float(np.sum((u - v)**2))
identity_val = float(2.0 * (1.0 - cos_theta))

print(f"Cosine Similarity: {cos_theta:.4f}")
print(f"Squared Euclidean: {d_e_sq:.4f}")
print(f"2 * (1 - cos theta): {identity_val:.4f}")
np.testing.assert_allclose(d_e_sq, identity_val, atol=1e-5)
print("VALIDASI IDENTITAS GEOMETRIS: Lulus sempurna!")

# Evaluasi Metrik Seksi 3 Part B (4 Kueri Target Ranks: [1, 3, 2, 6])
ranks = np.array([1, 3, 2, 6])
r1 = float(np.mean(ranks <= 1))
r5 = float(np.mean(ranks <= 5))
mrr = float(np.mean(1.0 / ranks))

print(f"Recall@1: {r1:.4f} | Recall@5: {r5:.4f} | MRR: {mrr:.4f}")
np.testing.assert_allclose(r1, 0.2500, atol=1e-4)
np.testing.assert_allclose(r5, 0.7500, atol=1e-4)
np.testing.assert_allclose(mrr, 0.5000, atol=1e-4)
print("VALIDASI METRIK RETRIEVAL: Hasil komputasi presisi sesuai derivasi manual!")

In [ ]:
# 4. Eksperimen Pipeline Retrieval Dwiarah Sintetis
# Simulasikan database 100 citra dan 10 kueri teks (D=16)
np.random.seed(42)
database_img = l2_norm(np.random.randn(100, 16))
queries_txt = l2_norm(np.random.randn(10, 16))

# Komputasi matriks kesamaan dot-product: (10, 100)
sim_matrix = np.dot(queries_txt, database_img.T)
# Peringkat teratas k=5
top5_retrieved = np.argsort(-sim_matrix, axis=1)[:, :5]

print(f"Database Ukuran: {database_img.shape[0]} citra | Kueri: {queries_txt.shape[0]} teks")
print("Top-5 Rekomendasi Citra untuk Kueri 0:", top5_retrieved[0].tolist())
print("Skor Kesamaan Tertinggi:", sim_matrix[0, top5_retrieved[0]].round(4).tolist())
print("VERIFIKASI PENCARIAN VEKTOR: Lulus deterministik.")

In [ ]:
# 5. Spesifikasi Eksekusi Indexing FAISS / HNSW (Untuk GPU / Colab)
if HAS_TORCH:
    print("Mode PyTorch / GPU Aktif.")
    q_tensor = torch.tensor(queries_txt, dtype=torch.float32)
    db_tensor = torch.tensor(database_img, dtype=torch.float32)
    torch_sims = torch.matmul(q_tensor, db_tensor.T)
    print("PyTorch Matrix Sim Shape:", torch_sims.shape)
else:
    print("PANDUAN SISTEM PENCARIAN VEKTOR SKALA BESAR (FAISS & HNSW):")
    print("1. Pasang paket faiss-cpu atau faiss-gpu:")
    print("   !pip install faiss-cpu")
    print("2. Pembuatan Indeks HNSW (Hierarchical Navigable Small World):")
    print("   import faiss")
    print("   index = faiss.IndexHNSWFlat(dimension, 32)")
    print("   index.add(database_embeddings.astype('float32'))")
    print("   distances, indices = index.search(query_vectors, k=10)")
    print("3. Penskalaan Produksi: Vector DB (Pinecone, Qdrant, Milvus) untuk pencarian multimodal real-time.")

### 6. Rangkuman & Pembahasan Akademik
1. **Kesetaraan Metrik**: Normalisasi $\ell_2$ menyatukan metrik Euclidean dan kosinus, memungkinkan komputasi retrieval diakselerasi melalui perkalian matriks paralel.
2. **Evaluasi Terstandar**: Recall@K dan Mean Reciprocal Rank (MRR) memetakan kualitas posisi peringkat secara objektif.
3. **Tantangan Ruang Tinggi**: Menyadari keberadaan fenomena Modality Gap dan Hubness problem merupakan kunci optimasi mesin pencari visual modern.